In [ ]:
#Create a binomial tree
import numpy as np
import math
import pandas as pd

def simulate_binomial_tree(S0, r, sigma, T, num_steps):
    # time interval
    dt = T / num_steps

    # up factor and down factor
    up_factor = math.exp(sigma * math.sqrt(dt))
    down_factor = 1.0 / up_factor

    # Initialize
    stock_prices = np.zeros((num_steps + 1, num_steps + 1))
    stock_prices[0, 0] = S0

    for step in range(1, num_steps + 1):
        for i in range(step):
            stock_prices[i, step] = stock_prices[i, step - 1] * up_factor
        stock_prices[step, step] = stock_prices[step - 1, step - 1] * down_factor

    return stock_prices

#test
simulate_binomial_tree(100, 0.05, 0.2, 1, 3)

In [ ]:
def option_price(put_or_call, K, T, S0, r, sigma, american = True, num_steps = 1000):
    
    stock_prices = simulate_binomial_tree(S0, r, sigma, T, num_steps)
    dt = T / num_steps
    up_factor = math.exp(sigma * math.sqrt(dt))
    down_factor = 1.0 / up_factor
    p = (math.exp(r * dt) - down_factor) / (up_factor - down_factor)
    if put_or_call == 'call':
        phi = 1
    else:
        phi = -1
    v = np.maximum(phi * (stock_prices[:, num_steps] - K), 0.)

    discount = math.exp(-r * dt)

    for i in range(num_steps-1, -1, -1) :
        n_nodes = i+1
        v = discount * ( p * v[:n_nodes] + (1-p) * v[1:n_nodes+1] )
        if american:
            v = np.maximum(phi * (stock_prices[:n_nodes, i] - K), v)

    return v[0]




In [ ]:
option_price('put', 4000, 1/365, 3853.39, 0.042, 0.29135, american = False, num_steps = 1000)

In [ ]:
option_data = pd.read_csv('clean_data2023.csv')
option_data = option_data[option_data['DTE'] != 0]
# Iterate over the dataset and calculate option prices
prices = []
for index, row in option_data.iterrows():
    price = option_price(
        put_or_call='put',
        K=row['STRIKE'],
        T=(row['DTE'])/365,
        S0=row['UNDERLYING_LAST'],
        r=row['rf'],
        sigma=row['P_IV'],
        american=False,  # Important for European options
        num_steps=1000  # Or any other number of steps you want to use
    )
    prices.append(price)

# Add calculated prices to the dataframe
option_data['calculated_price'] = prices

# Optionally, save or display the results
print(option_data)